# E7 Notebook 27D — Recover saved WebAttacks source audits and continue

**Repair:** Two legitimate source scans agree exactly but used different missing-label sentinels (`__MISSING__` versus `__MISSING_LABEL__`). This notebook compares equivalent missingness categories only **in memory**. It never alters any original G0 or extended GitHub report, raw CSV, label, source checksum, or old benchmark result.

**Critical quality finding:** Original Thursday WebAttacks has 288,602 missing labels out of 458,968 rows; this requires independent source investigation. No labels are filled or assumed benign. The workflow stays a read-only data audit; no classifier training is authorized.

Retains the CI-tested byte-strict encoding checks, data source fingerprints, verified per-file GitHub uploads and stop-on-genuine-disagreement semantics from Notebook 27C.


## 1. Verified immutable code snapshot and all synthetic regression tests

Source modules are pinned to a specific GitHub commit and independently verified against recorded Git blob SHA hashes. This uses a new E7-only directory and never updates an existing Git checkout.

In [ ]:
from pathlib import Path
import sys, hashlib, subprocess, json
from urllib.request import urlopen

PINNED_COMMIT = '2a0efc7669f5b799c15b04a8cd18b676a875a820'
SOURCE_BLOBS = {
    "scripts/e7_g0_external_source_audit.py": "67edf100fa438addeadefba17a8b9295fb733d54",
    "scripts/e7_g0_publish.py": "0d7427d2eced1fba90c7b2b8ce29feadd443b22e",
    "scripts/e7_g0_publish_v2.py": "cef711e90a865c15e336598017b223155ff04032",
    "scripts/e7_g0_extended_audit.py": "7aced8bd3bd5bf55bd2c588538aca56e72f637ec",
    "scripts/e7_g0_external_source_audit_v2.py": "b435c79f3e7d8d37259a8a92af894a09566c7ac4",
    "scripts/e7_g0_extended_audit_v2.py": "781bcd71226da279868913bce61cdb259d6feb2d",
    "scripts/e7_csv_encoding_gate.py": "48458a03f33c5a3a7d5e9447c08cbcd6f7ceaa0e",
    "configs/e7_g0_timestamp_registry_001.json": "3e1c5f2e3252fa6923108eba02ac238b5a8561dc",
    "tests/test_e7_g0_external_source_audit.py": "bf028eb5bdf6d4968403c608c79299c20c5709de",
    "tests/test_e7_g0_publish.py": "dc7f52cce749d82e9b2dc12f63a3686303e97afe",
    "tests/test_e7_g0_timestamp_registry.py": "55f70329719d48b2186179966f514c9992056718",
    "tests/test_e7_g0_publish_v2.py": "99ec055ebfdcef8604db2b80bca5ae788032ce43",
    "tests/test_e7_g0_extended_audit.py": "963c041e6869a1df765779ac69e6a18e22874049",
    "tests/test_e7_csv_encoding_gate.py": "acd3e28d94cbe33a188ce235e5ea6c49a2f6dee3"
}
REPO_REMOTE='itsCodeBakery/rare-attack-spatiotemporal-ids'
RESEARCH_PREFIX='research/E7_CICIDS2017'
SOURCE_DIR=Path('/kaggle/working/E7_OVERNIGHT_002_IMMUTABLE_SOURCE')
STUDY=SOURCE_DIR/RESEARCH_PREFIX

def git_blob_sha(data):
    return hashlib.sha1(b'blob '+str(len(data)).encode()+b'\\0'+data).hexdigest()

# Use a normalized explicit NUL in the Git object header.
def verify_git_blob(data,expected):
    actual=hashlib.sha1(b'blob '+str(len(data)).encode()+bytes([0])+data).hexdigest()
    assert actual==expected,f'Wrong remote source revision: {actual} != {expected}'
    return True

for relative,blob_hash in SOURCE_BLOBS.items():
    local=STUDY/relative
    if local.exists():
        content=local.read_bytes()
        verify_git_blob(content,blob_hash)
    else:
        url=('https://raw.githubusercontent.com/'+REPO_REMOTE+'/'+PINNED_COMMIT+
             '/'+RESEARCH_PREFIX+'/'+relative)
        with urlopen(url,timeout=90) as response:
            content=response.read()
        verify_git_blob(content,blob_hash)
        local.parent.mkdir(parents=True,exist_ok=True)
        local.write_bytes(content)
print('PINNED SOURCE VERIFIED:',len(SOURCE_BLOBS),'files')
print('SOURCE COMMIT:',PINNED_COMMIT)
print('NO Git clone, merge, pull, reset or prior checkout modification')

test_paths=sorted((STUDY/'tests').glob('test_e7_g0_*.py')) + [STUDY/'tests/test_e7_csv_encoding_gate.py']
assert all(p.is_file() for p in test_paths), 'A required E7 synthetic test file is missing.'
assert len(test_paths)==6,f'Expected 6 test modules, found {len(test_paths)}'
subprocess.run([sys.executable,'-m','pytest','-q',*map(str,test_paths)],check=True)
print('PREFLIGHT PASSED — all synthetic tests green; original research preserved')

## 2. Preflight source and GitHub secret

Confirm the 8 exact input CSV names and per-file timestamp registry, then check GitHub credentials. This preflight reads only small timestamp previews, not full data; full strict file encoding is checked later, just before scanning each new source.

In [ ]:
import importlib.util
from kaggle_secrets import UserSecretsClient
from urllib.error import HTTPError
import pandas as pd

def import_module(name,path):
    definition=importlib.util.spec_from_file_location(name,str(path))
    module=importlib.util.module_from_spec(definition)
    definition.loader.exec_module(module)
    return module

audit=import_module('e7_audit',STUDY/'scripts/e7_g0_external_source_audit_v2.py')
publisher=import_module('e7_publisher_v2',STUDY/'scripts/e7_g0_publish_v2.py')
quality=import_module('e7_quality',STUDY/'scripts/e7_g0_extended_audit_v2.py')
encoding_gate=import_module('e7_encoding_gate',STUDY/'scripts/e7_csv_encoding_gate.py')
registry=json.loads((STUDY/'configs/e7_g0_timestamp_registry_001.json').read_text())
entries=registry['filename_registry']
DATA_ROOT=Path('/kaggle/input/datasets/dsshayan/dscicids')
assert DATA_ROOT.is_dir(), 'Attach the CIC-IDS2017 dataset before running.'
FILES=sorted(DATA_ROOT.rglob('*.csv'))
assert len(FILES)==8 and {p.name for p in FILES}==set(entries), 'External source file set changed.'
assert len({str(p.parent) for p in FILES})==1
assert FILES[0].parent.name=='TrafficLabelling ', 'Unexpected source directory.'
print('E7 SOURCE ROOT:',repr(str(FILES[0].parent)))

for p in FILES:
    sample=pd.read_csv(p,nrows=4,encoding='utf-8-sig',dtype='string',
                       usecols=lambda col:col.strip().lower()=='timestamp')
    assert sample.shape[1]==1, 'Missing original timestamp.'
    dates=pd.to_datetime(sample.iloc[:,0],format=entries[p.name]['timestamp_format'],
                         errors='coerce',exact=True)
    assert dates.notna().all(), 'Original timestamp sample fails registered parser.'
    assert set(dates.dt.day_name())=={entries[p.name]['expected_weekday']}

token=UserSecretsClient().get_secret('pushRare')
assert token, 'Set Kaggle pushRare secret before starting; never paste into outputs.'
publisher.api('GET','https://api.github.com/repos/'+REPO_REMOTE,token)
print('PREFLIGHT PASSED — dataset, timestamps and GitHub authentication available')
print('Existing earlier UNSW / M1 artifacts will not be read or changed.')

## 3. Recover existing work; continue full audits without altering originals

The already-completed five G0 and five extended reports are reverified against original CSV SHA-256. Remaining files are checked byte-for-byte under strict UTF-8, and under strict Windows-1252 only if UTF-8 fails. Non-UTF8 source text is flagged for label review. A G0 or extended report is uploaded and verified on GitHub before proceeding to the next file.

In [ ]:
import base64
from collections import Counter

OLD_G0=Path('/kaggle/working/E7-EXT-VALIDITY-001/G0')
QUALITY_DIR=Path('/kaggle/working/E7-OVERNIGHT-001/G0_EXTENDED')
RECEIPTS=Path('/kaggle/working/E7-OVERNIGHT-001/UPLOAD_RECEIPTS')
OLD_G0.mkdir(parents=True,exist_ok=True)
QUALITY_DIR.mkdir(parents=True,exist_ok=True)
RECEIPTS.mkdir(parents=True,exist_ok=True)
report_pairs=[]
source_encoding_for={}

def safe_existing_public(stage,basename,expected_source_hash):
    filename=basename+('.audit.json' if stage=='G0' else '.quality.json')
    url=('https://api.github.com/repos/'+publisher.REPO+'/contents/'+
         publisher.PREFIX+'/'+stage+'/'+filename)
    try:
        record=publisher.api('GET',url,token)
    except HTTPError as exc:
        if exc.code==404:return None
        raise
    raw=base64.b64decode(record['content'])
    obj=json.loads(raw)
    publisher.encode_safe(obj,stage)
    source_field='input_artifact_sha256' if stage=='G0' else 'source_sha256'
    assert obj[source_field]==expected_source_hash, 'Uploaded original SHA differs from current CSV'
    assert (obj['external_source_basename'] if stage=='G0' else obj['source_file'])==basename
    return obj

def atomic_original_or_equal(target,obj):
    data=json.dumps(obj,sort_keys=True,indent=2,ensure_ascii=False)+'\n'
    if target.exists():
        original=json.loads(target.read_text(encoding='utf-8'))
        assert original==obj,'Local checkpoint differs: do not overwrite original.'
    else:
        tmp=target.with_suffix(target.suffix+'.new')
        tmp.write_text(data,encoding='utf-8')
        tmp.replace(target)

def registered_g0(path,source_sha):
    fmt=entries[path.name]['timestamp_format']
    o=audit.inspect(path,timestamp_format=fmt,max_rows=0,chunk_size=40000,
                    source_encoding=source_encoding_for[path.name])
    assert o['input_artifact_sha256']==source_sha
    item=o['files'][0]
    known_weekday=entries[path.name]['expected_weekday']
    days={key.split(' | ',1)[0] for key in item['label_day_counts']}
    suspicious_days=[]
    for day in sorted(days):
        try:
            if pd.Timestamp(day).day_name()!=known_weekday:suspicious_days.append(day)
        except Exception:suspicious_days.append(day)
    if suspicious_days:
        o['explicit_followup_gates'].append(
            'Original date disagrees with expected filename weekday; review required.')
    if 'Afternoon' in path.name:
        o['explicit_followup_gates'].append(
            'CLOCK_AMPM_AMBIGUITY: Afternoon source has unqualified 1-11 hour in prior preview. '
            'The original clock convention has not been independently verified.')
    if suspicious_days or 'Afternoon' in path.name:
        item['status']='DATA_QUALITY_REVIEW_REQUIRED'
        o['status']='G0_NOT_CLEARED'
    return o

for number,p in enumerate(FILES,1):
    print('\n'+'='*67,flush=True)
    print(f'ORIGINAL E7 SOURCE [{number}/8] {p.name}',flush=True)
    source_sha=audit.fingerprint(p)
    # Do NOT re-decode finished and already stored original audits.
    # Their old UTF-8 readers scanned every original row successfully.
    base_path=OLD_G0/(p.name+'.audit.json')
    extended_path=QUALITY_DIR/(p.name+'.quality.json')
    original_complete=(base_path.exists() and extended_path.exists())
    if original_complete:
        source_encoding_for[p.name]='utf-8-sig'
        print('Encoding was already validated by both completed original UTF-8 scans.',flush=True)
    else:
        print('Checking strict original-byte text encoding (may take minutes)...',flush=True)
        info=encoding_gate.source_encoding_gate(p)
        assert info['original_csv_sha256']==source_sha
        source_encoding_for[p.name]=info['source_encoding']
        print('ENCODING:',info['encoding_status'],'| review required:',info['requires_manual_review'],flush=True)

    if base_path.exists():
        basic=json.loads(base_path.read_text())
        assert basic['external_source_basename']==p.name
        assert basic['input_artifact_sha256']==source_sha
        assert basic['timestamp_format_declared']==entries[p.name]['timestamp_format']
        publisher.encode_safe(basic,'G0')
        print('REUSING source-checked existing original G0 checkpoint.',flush=True)
    else:
        basic=safe_existing_public('G0',p.name,source_sha)
        if basic is None:
            basic=registered_g0(p,source_sha)
            print('Completed full G0 original source scan.',flush=True)
        else:
            assert basic['timestamp_format_declared']==entries[p.name]['timestamp_format']
            print('RECOVERED exact original G0 result from GitHub.',flush=True)
        atomic_original_or_equal(base_path,basic)

    first=publisher.publish(base_path,token=token,stage='G0',
                            receipts_dir=RECEIPTS)
    print('GITHUB VERIFIED BASE:',first['status'],first['verified_via'],flush=True)

    if extended_path.exists():
        ext=json.loads(extended_path.read_text())
        assert ext['source_sha256']==source_sha
        publisher.encode_safe(ext,'G0_EXTENDED')
        print('REUSING exact source-checked extended checkpoint.',flush=True)
    else:
        ext=safe_existing_public('G0_EXTENDED',p.name,source_sha)
        if ext is None:
            print('SCANNING ALL ROWS for label, duration, feature,'
                  ' duplicate and clock quality...',flush=True)
            ext=quality.audit_file(
                p,timestamp_format=entries[p.name]['timestamp_format'],
                expected_weekday=entries[p.name]['expected_weekday'],chunk_size=35000,
                source_encoding=source_encoding_for[p.name])
            print('Extended original data-quality scan complete.',flush=True)
        else:
            print('RECOVERED original extended quality from GitHub.',flush=True)
        atomic_original_or_equal(extended_path,ext)

    second=publisher.publish(extended_path,token=token,
                             stage='G0_EXTENDED',receipts_dir=RECEIPTS)
    print('GITHUB VERIFIED EXTENDED:',second['status'],second['verified_via'],flush=True)
    assert basic['total_rows_inspected']==ext['rows'], 'Original vs extended row count drift'
    # Source-only audit modules use different literal missing-label sentinels.
    # Compare counts only after exact sentinel canonicalization in memory.
    # Never change original published reports, missing rows, or real labels.
    def audit_label_counts(d):
        counts={}
        for label,n in d.items():
            name='__MISSING_LABEL__' if label in ('__MISSING__','__MISSING_LABEL__') else label
            counts[name]=counts.get(name,0)+int(n)
        return counts
    assert audit_label_counts(basic['files'][0]['labels'])==audit_label_counts(ext['labels']), (
        'True source label-count disagreement. Original reports preserved; stop for investigation.'
    )
    if any(k in basic['files'][0]['labels'] for k in ('__MISSING__','__MISSING_LABEL__')):
        print('QUALITY FLAG: source contains missing labels; no imputation or relabeling.',flush=True)
    if source_encoding_for[p.name]=='cp1252':
        assert basic['status']=='G0_NOT_CLEARED', 'Strict non-UTF8 source must remain review-required'
    report_pairs.append((basic,ext))
    print('ROWS:',ext['rows'],
          '| Classes:',len(ext['labels']),
          '| Candidate duplicate rows:',ext['duplicates']['candidate_duplicate_rows'],
          '| Candidate contradictory labels:',
           ext['duplicates']['candidate_label_conflict_rows'])
    print('CLOCK AMBIGUOUS:',ext['timestamp_quality']['clock_ambiguous'],
          '| G0 STATUS:',basic['status'],flush=True)
    print('GITHUB RESEARCH REPORTS:',
          'https://github.com/'+publisher.REPO+'/tree/main/'+publisher.PREFIX)

print('\nE7 PER-SOURCE BACKUPS: ALL',len(report_pairs),'OF 8 ARCHIVED + VERIFIED.')
print('NO MODEL TRAINING. NO ORIGINAL UNSW TEST. TIME VALIDITY NOT CLEARED.')

## 4. Immutable GitHub summary after all eight sources

Publishes the eight-source comprehensive *data-quality* and provenance summary only after each per-file report is verified on GitHub. It is not an IDS model benchmark.

In [ ]:
from collections import Counter
import time

assert len(report_pairs)==8, 'Finish all eight source audits before publishing summary.'
class_counts=Counter()
source_items=[]
for basic,extended in report_pairs:
    class_counts.update(extended['labels'])
    source_items.append({
       'file':extended['source_file'],
       'sha256':extended['source_sha256'],
       'rows':extended['rows'],
       'classes':len(extended['labels']),
       'g0_status':basic['status'],
       'clock_ambiguous':extended['timestamp_quality']['clock_ambiguous'],
       'timestamp_parse_failures':extended['timestamp_quality']['parse_failures'],
       'source_row_order_reversals':extended['timestamp_quality']['row_order_reversals'],
       'hash_candidate_duplicate_rows':extended['duplicates']['candidate_duplicate_rows'],
       'hash_candidate_conflicting_label_rows':extended['duplicates']['candidate_label_conflict_rows']})
source_items.sort(key=lambda obj:obj['file'])
summary={
  'schema':'E7_OVERNIGHT_FULL_SOURCE_SUMMARY_001',
  'stage':'READ_ONLY_ORIGINAL_CICIDS2017_G0_AND_EXTENDED',
  'run_id':'E7-OVERNIGHT-001',
  'source_code_pinned_commit':PINNED_COMMIT,
  'files_verified':len(report_pairs),
  'rows_total':sum(x['rows'] for x in source_items),
  'labels_total':dict(sorted(class_counts.items())),
  'sources':source_items,
  'all_g0_cleared':False,
  'causal_traffic_order_proven':False,
  'original_unsw_test_accessed':False,
  'model_training_executed':False,
  'scientific_status':'DATA_QUALITY_AND_CLOCK_SEMANTICS_UNRESOLVED',
  'limitations':[
    'Original CICIDS2017 labels are not independently corrected or validated.',
    'Afternoon source clock values lack AM/PM and may not be true 24-hour times.',
    'Flow Duration units and flow-start/end causality remain unverified.',
    'Candidate duplicates use 64-bit hash and cannot establish exact leakage.',
    'No chronological external model evaluation or novelty proof was performed.',
    'No comparison to publisher 2017 Wednesday within-dataset benchmark is claimed.'
  ]
}
summary_data=(json.dumps(summary,indent=2,sort_keys=True,ensure_ascii=False)+'\n').encode()
summary_path=Path('/kaggle/working/E7-OVERNIGHT-001/G0_OVERNIGHT_SUMMARY.json')
if summary_path.exists():
    assert summary_path.read_bytes()==summary_data, 'Original summary conflicts: refuse overwrite'
else:
    summary_path.parent.mkdir(parents=True,exist_ok=True)
    summary_path.write_bytes(summary_data)
remote_name=(publisher.PREFIX+'/G0_EXTENDED/E7_OVERNIGHT_001_SUMMARY.json')
url=('https://api.github.com/repos/'+publisher.REPO+'/contents/'+remote_name)
digest=hashlib.sha256(summary_data).hexdigest()
receipt_path=RECEIPTS/'OVERNIGHT_001_SUMMARY_RECEIPT.json'
stored=None
if receipt_path.exists():
    stored=json.loads(receipt_path.read_text())
    assert stored.get('local_sha256')==digest and stored.get('remote')==remote_name, \
        'Summary original upload receipt mismatch; do not overwrite.'
    blob=stored.get('blob_sha')
    commit=stored.get('commit_sha')
else:
    blob=commit=None

verified=None
for seconds in (0,1,2,3,5,8):
    if seconds:time.sleep(seconds)
    verified=publisher.check_remote(url,token,summary_data,blob=blob,commit=commit)
    if verified:break
if verified is None and stored is None:
    # Create-only! No update SHA, and no retry PUT on a 404.
    response=publisher.api('PUT',url,token,{
        'message':'research(e7): verified eight-file original source audit summary',
        'branch':'main','content':base64.b64encode(summary_data).decode('ascii')})
    blob=response['content']['sha']
    commit=response['commit']['sha']
    publisher.atomic_json(receipt_path,{'local_sha256':digest,'remote':remote_name,
                                        'blob_sha':blob,'commit_sha':commit,'status':'PENDING_VERIFY'})
    for seconds in (0,1,2,3,5,8,13):
        if seconds:time.sleep(seconds)
        verified=publisher.check_remote(url,token,summary_data,blob=blob,commit=commit)
        if verified:break
if not verified:
    raise RuntimeError('GitHub summary may already be uploaded. Receipt saved. Stop; no blind PUT.')
publisher.atomic_json(receipt_path,{'local_sha256':digest,'remote':remote_name,
    'blob_sha':blob or verified['blob_sha'],'commit_sha':commit,
    'status':'VERIFIED','verified_via':verified['via']})

print('\n'+'='*74)
print('OVERNIGHT E7 RESULTS BACKED UP AND GITHUB VERIFIED')
print('Original CSV files:',len(report_pairs),
      '| All source rows:',summary['rows_total'])
print('Class counts:',summary['labels_total'])
print('Source files requiring review:',
      sum(x['g0_status']=='G0_NOT_CLEARED' for x in source_items))
print('Ambiguous clock files:',
      [x['file'] for x in source_items if x['clock_ambiguous']])
print('SUMMARY URL: https://github.com/'+publisher.REPO+'/blob/main/'+remote_name)
print('NO IDS MODELS TRAINED. EXTERNAL VALIDATION / CAUSAL ORDER REMAIN PENDING.')
print('='*74)

## When you wake up

Check the GitHub report links from the last cell. Each G0 result preserves original source hashes and, where needed, documents strict Windows-1252 decoding in its follow-up gates; corrected labeling/flow time remains unresolved. No model fitting, no invented external accuracy, no previously frozen UNSW/M1 data changed.